In [0]:
import dlt
from pyspark.sql.functions import col, lit, concat, trim, upper, when, coalesce

In [0]:
@dlt.table(name="silver_violations_combined")
def silver_violations_combined():
    chicago = (
        spark.table("workspace.default.silver_chicago_inspections")
        .where(col("violation_code").isNotNull() & (col("violation_code") != ""))
        .select("source_city", "violation_code", "violation_description", "violation_detail", "violation_severity")
    )
    
    dallas = (
        spark.table("workspace.default.silver_dallas_inspections")
        .where(col("violation_code").isNotNull() & (col("violation_code") != ""))
        .select("source_city", "violation_code", "violation_description", "violation_detail", "violation_severity")
    )
    
    return chicago.union(dallas).dropDuplicates(["violation_code", "source_city", "violation_description"])

In [0]:
dlt.create_target_table(
    name="dim_violation_scd2",
    comment="SCD Type 2 dimension for violation codes - tracks description changes over time",
    table_properties={"quality": "gold"}
)

dlt.apply_changes(
    target="dim_violation_scd2",
    source="silver_violations_combined",
    keys=["violation_code", "source_city"],
    sequence_by=lit(1),
    stored_as_scd_type=2,
    track_history_column_list=["violation_description", "violation_detail", "violation_severity"]
)